## Free-Wilson Analysis

Free-Wilson analysis is a classic **quantitative structure-activity relationship (QSAR)** method introduced by Spencer Free and James Wilson in 1964 [[1]](#ref1). The core idea is to treat the biological activity of a molecule as an **additive sum of contributions** from individual substituents at defined positions on a common molecular scaffold.

---

### References

<a id="ref1"></a>
[1] Free, S. M.; Wilson, J. W. A Mathematical Contribution to Structure-Activity Studies. *J. Med. Chem.* **1964**, *7* (4), 395–399. DOI: [10.1021/jm00334a001](https://doi.org/10.1021/jm00334a001)

## 1. Imports and Configuration

In [ ]:
import os
import itertools
from collections import Counter
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import AllChem, Draw, PandasTools
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdRGroupDecomposition
from rdkit.Chem.rdRGroupDecomposition import RGroupDecompositionParameters

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Misc Settings
pd.set_option("display.max_colwidth", 60)
RDLogger = Chem.rdBase
Chem.rdBase.DisableLog("rdApp.warning")  # silence noisy RDKit warnings

# ---- Free-Wilson Configuration ---- #
CSV_PATH = "[input].csv"            # path to input CSV
SMILES_COLUMN = "SMILES"            # column containing SMILES strings
ACTIVITY_FIELD = "pIC50"            # Activity column (log-scale)
OUTPUT_CSV = "[output].csv"         # path to output 
TOP_N_RGROUPS_PER_POSITION = 3      # how many best substituents to combine at each R-position
MAX_NEW_COMPOUNDS = 500             # cap on virtual compounds written to CSV
RANDOM_STATE = 42

In [ ]:
def load_csv(csv_path: str, smiles_column: str, activity_field: str) -> pd.DataFrame:
    """Load a CSV into a DataFrame with columns ['ROMol', 'SMILES', activity_field].

    Drops rows with unparseable SMILES or non-numeric activity values.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError("CSV parsed to an empty DataFrame.")

    for col in (smiles_column, activity_field):
        if col not in df.columns:
            raise KeyError(
                f"Column '{col}' not present in CSV. "
                f"Available columns: {list(df.columns)}"
            )

    # Parse SMILES -> RDKit Mol
    df["ROMol"] = df[smiles_column].apply(
        lambda s: Chem.MolFromSmiles(s) if isinstance(s, str) else None
    )
    n_before = len(df)
    df = df.dropna(subset=["ROMol"]).copy()
    if len(df) < n_before:
        print(f"Dropped {n_before - len(df)} rows with unparseable SMILES.")

    # Coerce activity to numeric
    df[activity_field] = pd.to_numeric(df[activity_field], errors="coerce")
    n_before = len(df)
    df = df.dropna(subset=[activity_field]).reset_index(drop=True)
    if len(df) < n_before:
        print(f"Dropped {n_before - len(df)} rows with non-numeric activity.")

    # Canonicalise SMILES (in case input had varied representations)
    df["SMILES"] = df["ROMol"].apply(Chem.MolToSmiles)
    print(f"Loaded {len(df)} compounds with valid activity values.")
    return df

df = load_csv(CSV_PATH, SMILES_COLUMN, ACTIVITY_FIELD)
df.head()

## 3. Find the Most Common Scaffold

We compute the **Bemis–Murcko scaffold** of each compound and pick the one that occurs in the largest number of molecules. The Murcko scaffold is the set of all ring systems in a molecule plus the linker atoms connecting them (side chains stripped).

For congeneric series, the dominant scaffold is normally shared by the majority of compounds; molecules whose scaffold differs are excluded from the R-group decomposition step.

In [ ]:
def get_murcko_scaffold_smiles(mol: Chem.Mol) -> Optional[str]:
    """Return canonical SMILES of the Bemis-Murcko scaffold, or None on failure."""
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        if scaffold is None or scaffold.GetNumAtoms() == 0:
            return None
        return Chem.MolToSmiles(scaffold)
    except Exception:
        return None

df["scaffold_smiles"] = df["ROMol"].apply(get_murcko_scaffold_smiles)

scaffold_counts = Counter(df["scaffold_smiles"].dropna())
print("Top 5 scaffolds by frequency:")
for s, c in scaffold_counts.most_common(5):
    print(f"  {c:4d}  {s}")

most_common_scaffold_smiles, most_common_count = scaffold_counts.most_common(1)[0]
most_common_scaffold = Chem.MolFromSmiles(most_common_scaffold_smiles)
print(
    f"\nSelected scaffold ({most_common_count}/{len(df)} compounds, "
    f"{100*most_common_count/len(df):.1f}%):\n  {most_common_scaffold_smiles}"
)

Draw.MolToImage(most_common_scaffold, size=(350, 250))

## 4. R-Group Decomposition

We feed the dominant scaffold as a *query* into RDKit's `RGroupDecomposition`, which returns one column per R-group attachment point (`R1`, `R2`, …) and one row per molecule. Each cell contains the R-group as an RDKit `Mol` with a dummy atom marking the attachment point.

We canonicalise each R-group to a SMILES string for use as a categorical feature in the regression. Molecules that fail to match the scaffold are dropped.

In [ ]:
def rgroup_decompose(
    mols: List[Chem.Mol], scaffold: Chem.Mol
) -> Tuple[pd.DataFrame, List[int]]:
    """Run R-group decomposition.

    Returns
    -------
    decomp_df : DataFrame with columns ['Core', 'R1', 'R2', ...] of SMILES strings.
    matched_idx : list of original-row indices that matched the scaffold.
    """
    params = RGroupDecompositionParameters()
    params.removeHydrogensPostMatch = True
    params.onlyMatchAtRGroups = False  # allow decomposition at any non-scaffold attachment

    decomp, unmatched = rdRGroupDecomposition.RGroupDecompose(
        [scaffold], mols, asSmiles=False, asRows=True, options=params
    )
    matched_idx = [i for i in range(len(mols)) if i not in set(unmatched)]

    rows = []
    for entry in decomp:
        rows.append({k: Chem.MolToSmiles(v) for k, v in entry.items()})
    decomp_df = pd.DataFrame(rows)
    return decomp_df, matched_idx

decomp_df, matched_idx = rgroup_decompose(df["ROMol"].tolist(), most_common_scaffold)

# Subset original DataFrame to molecules that matched the scaffold, align indices.
df_matched = df.iloc[matched_idx].reset_index(drop=True)
decomp_df = decomp_df.reset_index(drop=True)

assert len(df_matched) == len(decomp_df), "Mismatch between decomposition and source rows"

rgroup_columns = [c for c in decomp_df.columns if c.startswith("R")]
print(
    f"Matched scaffold on {len(df_matched)}/{len(df)} compounds; "
    f"found {len(rgroup_columns)} R-positions: {rgroup_columns}"
)
decomp_df.head()

### 4a. Inspect the R-group diversity at each position

In [ ]:
for col in rgroup_columns:
    counts = Counter(decomp_df[col])
    print(f"\n{col}: {len(counts)} unique substituents")
    for smi, n in counts.most_common(5):
        print(f"  {n:4d}  {smi}")

## 5. Free-Wilson Regression

To avoid perfect multicollinearity (the indicators at each position sum to 1), we drop one **reference substituent** per position (the most common one), so coefficients represent the change in activity relative to that reference.

We fit:
- An ordinary least squares (OLS) model for the headline coefficients and $R^2$.
- A k-fold cross-validated prediction to report $Q^2$ (a more honest generalization estimate).

When the design matrix is near-singular (very small datasets or unbalanced R-groups), we fall back to a small ridge penalty.


In [ ]:
def build_design_matrix(
    decomp_df: pd.DataFrame, rgroup_columns: List[str]
) -> Tuple[pd.DataFrame, Dict[str, str]]:
    """Build a one-hot design matrix, dropping the most common substituent per position
    as the reference category.

    Returns the X DataFrame and a dict {position: reference_smiles}.
    """
    references = {}
    one_hot_blocks = []
    for col in rgroup_columns:
        ref = decomp_df[col].mode().iloc[0]
        references[col] = ref
        dummies = pd.get_dummies(decomp_df[col], prefix=col, prefix_sep="=").astype(int)
        ref_col = f"{col}={ref}"
        if ref_col in dummies.columns:
            dummies = dummies.drop(columns=[ref_col])
        one_hot_blocks.append(dummies)
    X = pd.concat(one_hot_blocks, axis=1)
    return X, references

X, references = build_design_matrix(decomp_df, rgroup_columns)
y = df_matched[ACTIVITY_FIELD].values

print(f"Design matrix shape: {X.shape}  (n_samples x n_features)")
print("Reference (held-out) substituent at each position:")
for pos, ref in references.items():
    print(f"  {pos}: {ref}")

In [ ]:
def fit_free_wilson(X: pd.DataFrame, y: np.ndarray) -> Tuple[object, Dict[str, float]]:
    """Fit a Free-Wilson linear model and return the model plus a dict of metrics.

    Uses OLS if the system is well-conditioned, otherwise a small-alpha Ridge as a
    numerically stable fallback.
    """
    # Condition number on the design matrix
    try:
        cond = np.linalg.cond(X.values.astype(float))
    except np.linalg.LinAlgError:
        cond = np.inf

    if cond < 1e10 and X.shape[0] > X.shape[1]:
        model = LinearRegression()
        model_name = "OLS"
    else:
        model = Ridge(alpha=0.1, random_state=RANDOM_STATE)
        model_name = "Ridge(alpha=0.1) [fallback - matrix ill-conditioned or under-determined]"

    model.fit(X.values, y)
    y_pred = model.predict(X.values)

    n_splits = min(5, max(2, len(y) // 3))  # safe k for small data
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    y_cv = cross_val_predict(model, X.values, y, cv=cv)

    metrics = {
        "model": model_name,
        "n_samples": len(y),
        "n_features": X.shape[1],
        "R2": r2_score(y, y_pred),
        "Q2_cv": r2_score(y, y_cv),
        "RMSE": float(np.sqrt(mean_squared_error(y, y_pred))),
        "RMSE_cv": float(np.sqrt(mean_squared_error(y, y_cv))),
        "MAE": mean_absolute_error(y, y_pred),
        "MAE_cv": mean_absolute_error(y, y_cv),
        "intercept": float(model.intercept_),
    }
    return model, metrics, y_pred, y_cv

model, metrics, y_pred, y_cv = fit_free_wilson(X, y)

print("=== Free-Wilson regression metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:12s} : {v:.4f}")
    else:
        print(f"  {k:12s} : {v}")

### 5a. Observed vs. predicted plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, y_pred, alpha=0.7, label="Train fit")
ax.scatter(y, y_cv, alpha=0.7, marker="x", label="Cross-validated")
lims = [min(y.min(), y_pred.min(), y_cv.min()), max(y.max(), y_pred.max(), y_cv.max())]
ax.plot(lims, lims, "--", color="grey", linewidth=1)
ax.set_xlabel(f"Observed {ACTIVITY_FIELD}")
ax.set_ylabel(f"Predicted {ACTIVITY_FIELD}")
ax.set_title(
    f"Free-Wilson model  |  R² = {metrics['R2']:.3f}   Q² = {metrics['Q2_cv']:.3f}"
)
ax.legend()
plt.tight_layout()
plt.show()

### 5b. R-group coefficient table

Each coefficient is the activity contribution of that R-group **relative to the reference substituent** at the same position. Positive = activity-enhancing, negative = activity-reducing.

In [ ]:
def parse_feature_name(name: str) -> Tuple[str, str]:
    pos, smi = name.split("=", 1)
    return pos, smi

coef_rows = []
# Reference rows (coefficient = 0 by construction)
for pos, ref_smi in references.items():
    coef_rows.append({"position": pos, "rgroup_smiles": ref_smi,
                      "coefficient": 0.0, "is_reference": True})
for name, coef in zip(X.columns, model.coef_):
    pos, smi = parse_feature_name(name)
    coef_rows.append({"position": pos, "rgroup_smiles": smi,
                      "coefficient": float(coef), "is_reference": False})

coef_df = (pd.DataFrame(coef_rows)
           .sort_values(["position", "coefficient"], ascending=[True, False])
           .reset_index(drop=True))
coef_df

In [ ]:
# Bar plot of coefficients per position
positions = sorted(coef_df["position"].unique())
fig, axes = plt.subplots(1, len(positions), figsize=(5 * len(positions), 4), squeeze=False)
for ax, pos in zip(axes[0], positions):
    sub = coef_df[coef_df["position"] == pos].copy()
    sub = sub.sort_values("coefficient")
    colors = ["#bbbbbb" if r else ("#2ca02c" if c > 0 else "#d62728")
              for c, r in zip(sub["coefficient"], sub["is_reference"])]
    ax.barh(range(len(sub)), sub["coefficient"], color=colors)
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels([s if len(s) < 25 else s[:22] + "..." for s in sub["rgroup_smiles"]],
                       fontsize=8)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_title(f"{pos} coefficients")
    ax.set_xlabel(f"Δ {ACTIVITY_FIELD}")
plt.tight_layout()
plt.show()

## 6. Enumerate Novel Compounds

We pick the top-N most positive coefficients at each R-position (always including the reference for completeness if it's competitive) and generate the **Cartesian product** of these substituents. For each combination we:
1. Assemble a SMILES by attaching R-groups to the scaffold's dummy atoms.
2. Predict activity = intercept + sum of coefficients of chosen R-groups.
3. Skip any combination that exactly reproduces a training compound.

Compounds are sorted by predicted activity (descending) and capped at `MAX_NEW_COMPOUNDS`.

In [ ]:
def get_top_rgroups(coef_df: pd.DataFrame, position: str, top_n: int) -> pd.DataFrame:
    """Return the top_n highest-coefficient substituents at the given position
    (always including the reference substituent so its 0-contribution is considered)."""
    sub = coef_df[coef_df["position"] == position].sort_values("coefficient", ascending=False)
    top = sub.head(top_n).copy()
    ref_row = sub[sub["is_reference"]]
    if not ref_row.empty and ref_row.iloc[0]["rgroup_smiles"] not in top["rgroup_smiles"].values:
        top = pd.concat([top, ref_row], ignore_index=True)
    return top.reset_index(drop=True)

top_per_position = {pos: get_top_rgroups(coef_df, pos, TOP_N_RGROUPS_PER_POSITION)
                    for pos in positions}
for pos, tbl in top_per_position.items():
    print(f"\n{pos}: candidate substituents")
    print(tbl[["rgroup_smiles", "coefficient", "is_reference"]].to_string(index=False))

In [ ]:
def assemble_molecule(scaffold: Chem.Mol, rgroup_assignment: Dict[str, str]) -> Optional[Chem.Mol]:
    """Combine a scaffold and a dict of R-group SMILES into a single molecule.

    The R-group decomposition produced fragments where attachment points are encoded as
    isotope-labelled dummy atoms ([*:1], [*:2], ...). RDKit's molzip joins them on matching labels.
    """
    try:
        parts = [Chem.Mol(scaffold)]
        for _, smi in rgroup_assignment.items():
            m = Chem.MolFromSmiles(smi)
            if m is None:
                return None
            parts.append(m)
        combined = parts[0]
        for p in parts[1:]:
            combined = Chem.CombineMols(combined, p)
        zipped = Chem.molzip(combined)
        Chem.SanitizeMol(zipped)
        return zipped
    except Exception:
        return None

# The scaffold from R-group decomposition (with dummy labels) is in decomp_df['Core'].
core_smiles = decomp_df["Core"].mode().iloc[0]
core_mol = Chem.MolFromSmiles(core_smiles)
print("Labelled core used for assembly:", core_smiles)
Draw.MolToImage(core_mol, size=(350, 250))

In [ ]:
# Canonical SMILES of training compounds for novelty check
train_smiles = set(df_matched["ROMol"].apply(Chem.MolToSmiles))

# Map (position, smiles) -> coefficient for fast lookup
coef_lookup = {(r.position, r.rgroup_smiles): r.coefficient for r in coef_df.itertuples()}

candidates = list(itertools.product(
    *[top_per_position[p]["rgroup_smiles"].tolist() for p in positions]
))
print(f"Enumerating {len(candidates)} candidate R-group combinations...")

results = []
for combo in candidates:
    assignment = dict(zip(positions, combo))
    predicted = metrics["intercept"] + sum(coef_lookup[(p, s)] for p, s in assignment.items())
    new_mol = assemble_molecule(core_mol, assignment)
    if new_mol is None:
        continue
    new_smi = Chem.MolToSmiles(new_mol)
    is_novel = new_smi not in train_smiles
    row = {"SMILES": new_smi,
           f"predicted_{ACTIVITY_FIELD}": predicted,
           "is_novel": is_novel}
    for p, s in assignment.items():
        row[p] = s
    results.append(row)

results_df = (pd.DataFrame(results)
              .drop_duplicates(subset=["SMILES"])
              .sort_values(f"predicted_{ACTIVITY_FIELD}", ascending=False)
              .reset_index(drop=True))

print(f"Generated {len(results_df)} unique compounds "
      f"({results_df['is_novel'].sum()} novel, "
      f"{(~results_df['is_novel']).sum()} reproducing training).")
results_df.head(10)

## 7. Save Predicted Compounds to CSV

We export the **novel** (not present in training) compounds first, sorted by predicted activity, optionally followed by training-set hits. The CSV contains the SMILES, predicted activity, novelty flag, and the R-group assignment at each position.

In [ ]:
out_df = results_df.copy()
# Put novel compounds first, then keep ordering by predicted activity
out_df = out_df.sort_values(
    ["is_novel", f"predicted_{ACTIVITY_FIELD}"], ascending=[False, False]
)
out_df = out_df.head(MAX_NEW_COMPOUNDS).reset_index(drop=True)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(out_df)} compounds to {OUTPUT_CSV}")
out_df.head(15)

## 8. Visualize the Top Predicted Compounds

In [ ]:
top_view = out_df[out_df["is_novel"]].head(8)
mols = [Chem.MolFromSmiles(s) for s in top_view["SMILES"]]
legends = [f"{ACTIVITY_FIELD}={v:.2f}" for v in top_view[f'predicted_{ACTIVITY_FIELD}']]
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(260, 220), legends=legends)